# Lee, Dalley et al. (2023) on dandiset 000636

Paper: Lee BR, Dalley R, et al. "Signature morphoelectric properties of diverse GABAergic interneurons in the human neocortex." Science 382, eadf6484 (2023), doi:10.1126/science.adf6484.

Dataset: [DANDI:000636](https://dandiarchive.org/dandiset/000636/0.241120.0510), "Human interneuron patch-seq electrophysiology", version 0.241120.0510 (doi:10.48324/dandi.000636/0.241120.0510): 706 NWB files, one neuron each, 109 donors, 24.5 GB. This is the only published version; the paper cites no version (its Data and materials availability says only that data "will be made available through BIL, DANDI, and NeMO"), so the most recent published version is pinned. Its content equals the draft's (same asset count and size).

Claims: `claims/lee-2023.yaml`, written from the author manuscript of the version of record (PMC12229143) and the bioRxiv preprint (doi:10.1101/2022.11.08.515739). This run had the version-of-record text only (`paper/10.1126-science.adf6484.txt`); its supplementary legends are missing.

Labels and other outside data come from the paper's code repository, [AllenInstitute/human_patchseq_gaba](https://github.com/AllenInstitute/human_patchseq_gaba) at commit 5eb357a1cea246c7c847b5608a8fc7e87ea95b58: the per-cell metadata (labels, matched to files by `nwb_id`), the paper's feature table (used only to check the features computed here), the morphology feature table, the DBC calls, the channel gene list and the transcriptomes. The electrophysiology features are computed from the NWB files with IPFX.

| Claim | Data | Tested here |
|---|---|---|
| cell-metadata-link | labels | tested |
| dataset-composition | labels | tested |
| fig1b-more-genes-in-culture, figs1a-genes-across-days, fig1d-nms-no-difference, fig1c-marker-expression-correlated | code repository | tested |
| figs2d-seven-channel-genes | code repository | run; blocked (gene filter needs data not in the repository) |
| figs1-microglial-signature, fig1e-wfdc2-transcriptome-overlap, figs1de-mapping-unaffected, fig2a-patchseq-overlaps-snrnaseq, fig3-frzb-transcriptome-intermediate | code repository, snRNA-seq | blocked (Seurat in R; snRNA-seq reference) |
| fig5a-dbc-two-ttypes, figs7a-dbc-regions, figs7a-dbc-markers, fig4d-calb1-genes-by-morphology | labels, code repository | tested |
| figs2b-wfdc2-ephys-43-of-84, fig1l-width-sag-differ, figs2c-no-change-with-days | dataset | run; blocked (unreleased features; readings disagree) |
| fig1i, fig2b, fig2f, fig2g, fig2h, fig3c, fig3e, figs6, fig4c, fig5c, fig5d-dbc-ephys-intermingled, figs5 | dataset with labels | tested |
| fig1g, fig2c, fig2e, fig3g, fig1h, fig5d-dbc-morph-umap | morphology feature table | tested |
| fig4a-calb1-morphology-types, fig5b-merfish-layers, fig5e-bouton-density, fig6b-species-morphology | reconstructions, MERFISH, images, mouse | blocked |
| fig6c, fig6e, figs8b, figs9 | mouse data (Gouwens et al. 2020) | blocked |

The full extraction streams all 706 files and takes about 25 minutes.

In [ ]:
import time, zlib, resource, warnings, logging
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import lee2023 as L
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
KEY = "lee-2023"
def seed(claim, k=0):
    """Seed of a claim's generator (protocol, author step 3); k-th generator adds k."""
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k
SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
T0 = time.time()

## Data

The asset list of the pinned version, the dandiset metadata, and the paper's tables from its code repository at commit 5eb357a (labels only; claim cell-metadata-link).

In [ ]:
A = L.assets()
V = L.version_metadata()
meta = L.metadata_table()
fx_paper = L.code_table("LeeDalley_ephys_fx.csv")
morph = L.code_table("LeeDalley_morpho_features.csv")
dbc_tab = L.code_table("dbc_calb1_ADGRG6.csv")
FEATS84 = [c for c in fx_paper.columns if c != "specimen_id" and c not in L.CHIRP]
print("version", L.VERSION, "assets", len(A), "subjects", A.subject.nunique(), "sessions", A.session.nunique(),
      "total GB", round(A["size"].sum() / 1e9, 2))
print("archive summary:", {k: V["assetsSummary"].get(k) for k in ["numberOfFiles", "numberOfSubjects", "numberOfCells"]})
print("metadata rows", len(meta), "unique specimens", meta.specimen_id_x.nunique(), "| feature table rows", len(fx_paper),
      "| feature columns", fx_paper.shape[1] - 1, "of which chirp", len(L.CHIRP), "-> 84 =", len(FEATS84))

## Feature extraction

Every NWB file is streamed and its long square, short square and ramp sweeps read; sweeps failing the Methods' QC are dropped, and IPFX computes the features (helper `cell_features`). This takes about 25 minutes.

In [ ]:
t = time.time()
R = L.all_cell_features(A.asset_id.tolist(), workers=3)
R = A.merge(R, on="asset_id")
print(f"extraction {time.time() - t:.0f} s; peak memory of a worker {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")

In [ ]:
FEATS = [c for c in FEATS84 if c in R.columns]
print("of the 84 features,", len(FEATS), "computed;", len(set(FEATS84) - set(FEATS)), "not computed by public IPFX:",
      sorted(set(FEATS84) - set(FEATS)))
err = R.filter(regex="^error").notna()
print("files with a read error:", int(err.get("error", pd.Series(False)).sum()))
for c in err.columns:
    if c != "error":
        print(c, int(err[c].sum()), R[c].dropna().str.split(":").str[0].str[:60].value_counts().to_dict())
m1 = meta.drop_duplicates()
D = R.merge(m1, left_on="session", right_on="nwb_session", how="left")
D["has_row"] = D.specimen_id_x.notna()
D["subclass"] = D.Revised_subclass_label
D["ttype"] = D.Transcriptomic_type
D["ls_ok"] = D.n_long_square_qc > 0
E = D[D.has_row & D.ls_ok].copy()     # labelled cells with at least one QC-passing long square sweep
print("labelled files", int(D.has_row.sum()), "| with a QC-passing long square sweep", len(E),
      "| with rheobase", int(E.rheobase_i.notna().sum()), "| with sag", int(E.sag.notna().sum()))

## Data check

In [ ]:
print("Files and subjects")
print("  files", len(R), "subjects (sub- folders)", R.subject.nunique(), "| subject_id in file equals folder:", bool((R.subject_id == R.subject).all()),
      "| session_id equals file name:", bool((R.session_id == R.session).all()))
print("  species strings:", R.species.value_counts().to_dict(), "| institution:", R.institution.value_counts().to_dict())
print("  fields under /general:", R.general_fields.value_counts().to_dict())
print("  electrode location:", R.electrode_location.value_counts().to_dict())
print("  archive summary says numberOfCells", V["assetsSummary"].get("numberOfCells"), "for", len(R), "files, one cell per file")
print("Sampling rates of the analysed sweeps (Hz):", R.rates.astype(str).value_counts().to_dict())
print("  the Methods state 'digitized at 50 kHz'; files at 200 kHz by session date:")
R["year"] = R.session_start_time.str[:4]
print(pd.crosstab(R.year, R.rates.astype(str)))
print("Units (response|stimulus):", R.units.astype(str).value_counts().to_dict())
print("Sweeps per file: long square", R.n_long_square.describe()[["min", "50%", "max"]].to_dict(),
      "| after QC", R.n_long_square_qc.describe()[["min", "50%", "max"]].to_dict())
print("  sweeps dropped by QC: long square %d of %d, short square %d of %d, ramp %d of %d" % (
      (R.n_long_square - R.n_long_square_qc).sum(), R.n_long_square.sum(), (R.n_short_square - R.n_short_square_qc).sum(),
      R.n_short_square.sum(), (R.n_ramp - R.n_ramp_qc).sum(), R.n_ramp.sum()))
print("  sweeps dropped for bridge balance >= 15% of Rin (cells):", int((R.n_bridge15_dropped > 0).sum()))
print("Labelled cells by condition and subclass (labelled files with a QC-passing long square sweep)")
print(pd.crosstab(E.subclass, E.condition, margins=True))
print("Donors", E.Donor.nunique(), "| cells per donor median", E.groupby("Donor").size().median())
print("Ranges of the variables used")
for c in ["rheobase_i", "input_resistance", "sag", "vm_for_sag", "width_rheo", "width_short_square", "upstroke_downstroke_ratio_rheo",
          "peak_deltav_rheo", "latency_rheo", "avg_rate_hero", "fi_fit_slope", "v_baseline", "tau"]:
    x = E[c].dropna()
    print(f"  {c:32s} n={len(x):3d} min={x.min():.4g} median={x.median():.4g} max={x.max():.4g}")
print("Clock check: the first spike of the rheobase sweep follows stimulus onset; latency (s) over cells:",
      E.latency_rheo.describe()[["min", "50%", "max"]].round(4).to_dict(), "| negative:", int((E.latency_rheo < 0).sum()),
      "| beyond the 1 s step:", int((E.latency_rheo > 1.0).sum()))
print("  contrast: rate in the hero sweep (rheobase + 40 pA) against rheobase sweep, median",
      E.avg_rate_hero.median(), "against", E.avg_rate_rheo.median(), "spikes/s; higher in",
      int((E.avg_rate_hero > E.avg_rate_rheo).sum()), "of", int(E[["avg_rate_hero", "avg_rate_rheo"]].notna().all(1).sum()), "cells")
print("  hyperpolarizing sweep used for sag reaches", E.vm_for_sag.describe()[["min", "50%", "max"]].round(1).to_dict(), "mV (target -100 mV)")
key = ["v_baseline", "rheobase_i", "input_resistance", "tau", "sag", "width_short_square", "threshold_i_ramp"]
kk = R[key].astype(float).round(6).fillna(-1e9).map(repr).agg("|".join, axis=1)
dup = R[R[key].notna().sum(1) >= 4][kk[R[key].notna().sum(1) >= 4].duplicated(keep=False)]
print("Deposited twice? files sharing identical values of", key, ":", len(dup), dup.session.tolist())
print("  files with the same file size:", int(R["size"].duplicated(keep=False).sum()))
print("Days in culture of culture cells:", E[E.condition == "culture"].Days_In_culture.value_counts().sort_index().to_dict())

## cell-metadata-link

> The dandiset's NWB files can be matched to the paper's per-cell metadata and feature tables in its code repository, and features recomputed from the files agree with those tables.

Matched by `nwb_id` (the NWB session id). Reproduced when at least 90% of the `Has_ephys` rows are found and rheobase_i, input_resistance, sag and width_rheo are within 10% of the paper's table for at least 90% of cells with both values.

In [ ]:
ephys_rows = m1[m1.Has_ephys == True]
found = ephys_rows.nwb_session.isin(R.session)
print("metadata rows", len(meta), "(one duplicated row:", int(meta.duplicated().sum()), ") -> unique rows", len(m1))
print("Has_ephys rows", len(ephys_rows), "| with nwb_id", int(ephys_rows.nwb_session.notna().sum()),
      "| found in the dandiset", int(found.sum()), f"({found.mean():.1%})")
print("  Has_ephys rows without nwb_id:", int(ephys_rows.nwb_session.isna().sum()),
      "| nwb_id not in the dandiset:", int((ephys_rows.nwb_session.notna() & ~found).sum()))
print("rows without Has_ephys whose nwb_id is in the dandiset:", int(((m1.Has_ephys != True) & m1.nwb_session.isin(R.session)).sum()))
nores = R[~R.session.isin(m1.nwb_session.dropna())]
print("NWB files without a metadata row:", len(nores), "| their sessions:", nores.session.tolist())
J = D[D.has_row].merge(fx_paper, left_on="specimen_id_x", right_on="specimen_id", suffixes=("", "_paper"))
LINK = {}
rows = []
for c in FEATS:
    a, b = J[c].astype(float), J[c + "_paper"].astype(float)
    ok = a.notna() & b.notna()
    rel = ((a - b).abs() / b.abs().where(b != 0))[ok]
    rows.append(dict(feature=c, n=int(ok.sum()), within10=(rel <= 0.1).mean(), median_rel_diff=rel.median(),
                     missing_here=int((a.isna() & b.notna()).sum()), missing_paper=int((a.notna() & b.isna()).sum())))
agree = pd.DataFrame(rows).set_index("feature")
for c in ["rheobase_i", "input_resistance", "sag", "width_rheo"]:
    LINK[c] = agree.loc[c, "within10"]
print(agree.loc[["rheobase_i", "input_resistance", "sag", "width_rheo"]].round(3))
four = ["rheobase_i", "input_resistance", "sag", "width_rheo"]
ok4 = J[four].notna().all(1) & J[[f + "_paper" for f in four]].notna().all(1)
joint = np.all([((J[f] - J[f + "_paper"]).abs() / J[f + "_paper"].abs() <= 0.1) for f in four], axis=0)[ok4.values]
print(f"all four within 10% together: {joint.mean():.3f} of {ok4.sum()} cells (a stricter reading, shown only)")
print("Agreement over all computed features (fraction of cells within 10%):")
print(agree.within10.describe().round(3).to_dict())
print("features with fewer than 80% of cells within 10%:", agree[agree.within10 < 0.8].within10.round(2).to_dict())
link_ok = found.mean() >= 0.9 and min(LINK.values()) >= 0.9
print("RESULT link:", "reproduced" if link_ok else ("partial" if min(LINK.values()) >= 0.5 else "not reproduced"))

**Result: reproduced.** 684 of the 704 rows with electrophysiology (97.2%) have an NWB file, and rheobase, input resistance, sag and width at rheobase computed here are within 10% of the paper's table for 92.2%, 94.5%, 97.8% and 98.8% of cells. All four agree together for 90.6% of cells. 16 files have no metadata row, and the metadata table repeats one row. Some features computed here differ from the table more often (isi_adapt_ratio within 10% for 27% of cells, fi_fit_slope 41%), so those definitions are not the paper's.

## dataset-composition

> The study characterized 778 human GABAergic neurons in layers 2 to 6, covering 44 of the 45 GABAergic t-types, with electrophysiology from 75 LAMP5/PAX6, 99 VIP, 149 SST and 352 PVALB neurons.

Counted over the dandiset's files with a QC-passing long square sweep, labelled through the link, by (a) `Revised_subclass_label` and (b) `subclass_label`; 778 and 44 over all labelled cells. Cell-level QC (seal, access resistance) is not recomputed here.

In [ ]:
exp = {"LAMP5/PAX6/Other": 75, "VIP": 99, "SST": 149, "PVALB": 352}
print("neurons in the metadata table (unique specimens):", m1.specimen_id_x.nunique(), "| t-types:", m1.Transcriptomic_type.nunique())
print("layers of the labelled cells:", m1.Cortical_layer.value_counts(dropna=False).to_dict())
comp = pd.DataFrame({"expected": exp,
                     "a_revised": E.Revised_subclass_label.value_counts(),
                     "b_subclass_label": E.subclass_label.value_counts(),
                     "all_Has_ephys_rows_revised": m1[m1.Has_ephys == True].Revised_subclass_label.value_counts()})
comp["a_within"] = (comp.a_revised - comp.expected).abs() <= 0.1 * comp.expected
comp["b_within"] = (comp.b_subclass_label - comp.expected).abs() <= 0.1 * comp.expected
print(comp)
print("files counted", len(E), "| sum of expected", sum(exp.values()))
print("variant, cells with a rheobase computed here:", int(E.rheobase_i.notna().sum()), E[E.rheobase_i.notna()].Revised_subclass_label.value_counts().to_dict(),
      "| subclass_label:", E[E.rheobase_i.notna()].subclass_label.value_counts().to_dict())
print("labelled files without a QC-passing long square sweep:", int((D.has_row & ~D.ls_ok).sum()))
print("RESULT composition: reading a all within:", bool(comp.a_within.all()), "| reading b all within:", bool(comp.b_within.all()))

**Result: partially reproduced.** The metadata table holds 778 neurons in 44 t-types, as the paper says. Counted over the files here, reading (a) gives VIP 101, SST 158 and PVALB 363, within tolerance, but LAMP5/PAX6 64 against 75. Reading (b) gives LAMP5/PAX6 69 and VIP 96 within tolerance, and SST 231 and PVALB 290 outside it. Cells with a rheobase number 675, the paper's total, but by subclass they are 63, 99, 155 and 358.

## Transcriptomes from the paper's code repository

The claims on gene expression are `data: elsewhere`: the expression matrices are in the code repository at commit 5eb357a (`complete_patchseq_data_sets1/2.RData`, log2(CPM + 1), 778 cells), read here with the `rdata` package.

In [ ]:
X, metaP = L.transcriptomes()
print("expression matrix genes x cells", X.shape, "| value range", float(X.values.min()), float(X.values.max()))
cells_meta = m1.set_index("patched_cell_container")
print("expression columns found in the metadata table:", int(X.columns.isin(cells_meta.index).sum()), "of", X.shape[1])
T = cells_meta.loc[X.columns]
genes_detected_here = (X > 0).sum(axis=0)
print("genes detected (>0) agrees with the Genes_Detected column for", int((genes_detected_here.values == T.Genes_Detected.values).sum()), "of", len(T),
      "cells; median absolute difference", float(np.median(np.abs(genes_detected_here.values - T.Genes_Detected.values))))

## fig1b-more-genes-in-culture

> Patch-seq neurons from slices in culture (2 to 16 days) had more genes detected than neurons from acute slices.

Two-sided Mann-Whitney, culture against acute (`condition`); reproduced when the culture median is higher at p < 0.0001.

In [ ]:
g = pd.DataFrame({"genes": genes_detected_here.values, "condition": T.condition.values, "dic": T.Days_In_culture.values})
for name, col in [("genes > 0 in the matrix", "genes")]:
    a, c = g[g.condition == "acute"][col], g[g.condition == "culture"][col]
    u, p = stats.mannwhitneyu(c, a, alternative="two-sided")
    print(f"{name}: acute n={len(a)} median={a.median():.0f}, culture n={len(c)} median={c.median():.0f}, U={u:.0f}, p={p:.3g}")
a2, c2 = T[T.condition == "acute"].Genes_Detected, T[T.condition == "culture"].Genes_Detected
print("Genes_Detected column: medians", a2.median(), c2.median(), "p=%.3g" % stats.mannwhitneyu(c2, a2).pvalue)
print("RESULT fig1b:", bool(c.median() > a.median() and p < 1e-4))

**Result: reproduced.** Culture cells have more genes detected (median 12203 against 11259 genes above 0, p = 4.3e-10; Genes_Detected column 7318 against 6802, p = 1.6e-05).

## figs1a-genes-across-days

> The higher number of genes detected in culture is present at every day in culture sampled and stays consistent across days in culture.

Pearson's correlation of genes detected with days in culture over culture cells (holds when p >= 0.05), and the median at every day above the acute median.

In [ ]:
cg = g[g.condition == "culture"]
r, p = stats.pearsonr(cg.dic, cg.genes)
per_day = cg.groupby("dic").genes.agg(["median", "count"])
per_day["above_acute_median"] = per_day["median"] > a.median()
print(per_day.T)
print(f"acute median {a.median():.0f}; Pearson r={r:.3f}, p={p:.3g}, n={len(cg)}")
rs, ps = stats.spearmanr(cg.dic, cg.genes)
print(f"variant, Spearman rho={rs:.3f}, p={ps:.3g}")
print("RESULT figs1a: consistency (p>=0.05):", bool(p >= 0.05), "| elevation at every day:", bool(per_day.above_acute_median.all()))

**Result: reproduced.** The median at every day in culture, 2 to 16, is above the acute median (11259), and genes detected do not correlate with days in culture (Pearson r = 0.029, p = 0.54, n = 467).

## fig1d-nms-no-difference

> The normalized marker sum score does not differ between acute and culture neurons.

A claim of absence, judged on the two-sided Mann-Whitney test of `marker_sum_norm_label`, acute against culture.

In [ ]:
nms_a = m1[m1.condition == "acute"].marker_sum_norm_label.dropna()
nms_c = m1[m1.condition == "culture"].marker_sum_norm_label.dropna()
u, p = stats.mannwhitneyu(nms_a, nms_c, alternative="two-sided")
print(f"acute n={len(nms_a)} median={nms_a.median():.3f}; culture n={len(nms_c)} median={nms_c.median():.3f}; U={u:.0f} p={p:.3g}; "
      f"rank-biserial={1 - 2 * u / (len(nms_a) * len(nms_c)):.3f}")
print("NMS range", round(m1.marker_sum_norm_label.min(), 3), round(m1.marker_sum_norm_label.max(), 3))
print("RESULT fig1d: no difference (p>=0.05):", bool(p >= 0.05))

**Result: reproduced.** NMS does not differ (medians 0.962 and 0.962, Mann-Whitney p = 0.38, rank-biserial -0.037).

## fig1c-marker-expression-correlated

> For the top differentially expressed genes of each GABAergic subclass, average expression in acute and culture neurons is strongly correlated.

Readings named by the criterion: Pearson or Spearman, log2(CPM + 1) or CPM, per subclass or pooled, genes ranked by differential expression in the Patch-seq cells (the snRNA-seq ranking needs the dissociated-nuclei reference, which the code repository does not hold: its notebook loads `complete_facs_data_sets.RData`, absent at 5eb357a). Patch-seq ranking: the 25 genes with the largest difference in mean log2(CPM + 1) between a subclass's cells and all other cells (a standard choice).

In [ ]:
sub_T = T.Revised_subclass_label.values
rows = []
top = {}
for s in ["LAMP5/PAX6/Other", "PVALB", "SST", "VIP"]:
    ins = sub_T == s
    diff = X.loc[:, ins].mean(axis=1) - X.loc[:, ~ins].mean(axis=1)
    top[s] = diff.sort_values(ascending=False).index[:25]
pts = {}
for scale in ["log2", "cpm"]:
    for s in top:
        ins = sub_T == s
        Y = X.loc[top[s]]
        if scale == "cpm":
            Y = 2 ** Y - 1
        ma = Y.loc[:, ins & (T.condition.values == "acute")].mean(axis=1)
        mc = Y.loc[:, ins & (T.condition.values == "culture")].mean(axis=1)
        pts[(scale, s)] = (ma, mc)
        for test, fn in [("pearson", stats.pearsonr), ("spearman", stats.spearmanr)]:
            r, p = fn(ma, mc)
            rows.append(dict(scale=scale, panel=s, test=test, r=r, p=p, n=len(ma)))
    ma = pd.concat([pts[(scale, s)][0] for s in top]); mc = pd.concat([pts[(scale, s)][1] for s in top])
    for test, fn in [("pearson", stats.pearsonr), ("spearman", stats.spearmanr)]:
        r, p = fn(ma, mc)
        rows.append(dict(scale=scale, panel="pooled", test=test, r=r, p=p, n=len(ma)))
c1 = pd.DataFrame(rows)
c1["holds"] = (c1.p < 1e-4) & (c1.r > 0)
print(c1.round(6).to_string())
per_sub = c1[c1.panel != "pooled"].groupby(["scale", "test"]).holds.all()
pooled = c1[c1.panel == "pooled"].set_index(["scale", "test"]).holds
print("per-subclass readings hold in every subclass:", per_sub.to_dict()); print("pooled readings:", pooled.to_dict())
print("cells acute/culture per subclass:", pd.crosstab(T.Revised_subclass_label, T.condition).to_dict())
print("RESULT fig1c: all readings run hold:", bool(per_sub.all() and pooled.all()))

**Result: reproduced, under an assumption.** Every reading run gives r > 0 and p < 1e-4, per subclass (r 0.75 to 0.97) and pooled (r 0.87 to 0.92). The genes are ranked here in the Patch-seq cells; the snRNA-seq ranking could not be run, as the reference data are not in the repository.

## figs2d-seven-channel-genes

> Of 114 voltage-gated ion-channel genes, only seven differ in expression between culture and acute neurons.

As the code's chunks "linear model gene expression WFDC2" and "fold change and ion channels WFDC2": PVALB WFDC2 cells, genes with mean > 0.01, `lm(expression ~ condition + isImmune + condition*isImmune)` per gene, `apply(lm_pval, 2, p.adjust)` (Holm over each gene's three term p-values, as the code applies it per gene), channel genes from `vgic_genes.csv`, counted at adjusted p <= 0.01. isImmune is the paper's call (`condition2` ending in "microglial"). The code's `keepGenes` filter needs the dissociated-nuclei data, not in the repository, so it is not applied.

In [ ]:
vgic = L.code_table("vgic_genes.csv")
channels = set(vgic["HGNC symbol"].dropna())
wf = (T.Transcriptomic_type == "Inh L2-4 PVALB WFDC2").values
Xw = X.loc[:, wf]
keep = Xw.mean(axis=1) > 0.01
Xw = Xw[keep]
cond = (T.condition.values[wf] == "culture").astype(float)
imm = T.condition2.str.contains("microglial").values[wf].astype(float)
Dm = np.column_stack([np.ones_like(cond), cond, imm, cond * imm])
print("PVALB WFDC2 cells", int(wf.sum()), "| by condition2", pd.Series(T.condition2.values[wf]).value_counts().to_dict())
Y = Xw.values.T
beta, *_ = np.linalg.lstsq(Dm, Y, rcond=None)
resid = Y - Dm @ beta
dof = Dm.shape[0] - np.linalg.matrix_rank(Dm)
s2 = (resid ** 2).sum(0) / dof
XtXi = np.linalg.pinv(Dm.T @ Dm)
se = np.sqrt(np.outer(np.diag(XtXi), s2))
with np.errstate(divide="ignore", invalid="ignore"):
    tval = beta / se
pv = 2 * stats.t.sf(np.abs(tval), dof)[1:]          # condition, isImmune, interaction
from statsmodels.stats.multitest import multipletests
def holm_cols(P):
    """R's p.adjust(method = "holm") applied to each column of P (here, each gene's three term
    p-values), vectorized: sorted p times (m - rank), cumulative maximum, capped at 1."""
    m_ = P.shape[0]
    order = np.argsort(P, axis=0)
    ps = np.take_along_axis(P, order, axis=0)
    adj_s = np.minimum(1, np.maximum.accumulate(ps * (m_ - np.arange(m_))[:, None], axis=0))
    out = np.empty_like(P)
    np.put_along_axis(out, order, adj_s, axis=0)
    return out
chk = np.array([[0.01, 0.2], [0.04, 0.03], [0.03, 0.5]])
print("check against statsmodels on two columns:", np.allclose(holm_cols(chk), np.column_stack([multipletests(chk[:, j], method="holm")[1] for j in range(2)])))
adj = holm_cols(pv)[0]
genes = Xw.index
isch = genes.isin(channels)
res = pd.DataFrame({"p_condition": pv[0], "adj_per_gene": adj, "channel": isch}, index=genes)
res["holm_across_genes"] = L.bh(res.p_condition) * np.nan
okp = np.isfinite(res.p_condition)
res.loc[okp, "holm_across_genes"] = multipletests(res.p_condition[okp], method="holm")[1]
res.loc[okp, "bh_across_genes"] = multipletests(res.p_condition[okp], method="fdr_bh")[1]
res["FC"] = Xw.loc[:, cond == 1].mean(1) - Xw.loc[:, cond == 0].mean(1)
chan = res[res.channel]
print("genes tested", len(res), "| channel genes in vgic list", len(channels), "| channel genes tested", len(chan))
n_main = int((chan.adj_per_gene <= 0.01).sum())
print("channel genes with adjusted p <= 0.01 (code's per-gene Holm):", n_main)
print(chan[chan.adj_per_gene <= 0.01].sort_values("adj_per_gene")[["p_condition", "adj_per_gene", "FC"]].round(5))
print("variants: Holm across genes", int((chan.holm_across_genes <= 0.01).sum()), "| BH across genes", int((chan.bh_across_genes <= 0.01).sum()),
      "| unadjusted p <= 0.01", int((chan.p_condition <= 0.01).sum()))
print("RESULT figs2d: count", n_main, "channel genes tested", len(chan))

**Result: blocked.** The code's gene filter needs the dissociated-nuclei data, which the repository does not hold. Without it 142 channel genes are tested, against the paper's 114, and 22 differ at adjusted p <= 0.01. The procedure run is therefore not the paper's, and the count is not compared with 7.

## fig5a-dbc-two-ttypes

> Morphologically classic double bouquet cells map to two t-types, SST CALB1 and SST ADGRG6.

DBC calls from `dbc_calb1_ADGRG6.csv`; their t-types there (`Tree_first_cl`) against the tree mapping stored with the transcriptomes (`Tree_first_cl` of `metaPatch`) and the metadata table. The tree mapping itself is not re-run (it needs the MTG reference taxonomy).

In [ ]:
dbc_yes = dbc_tab[dbc_tab.DBC == "yes"].copy()
mp = metaP.set_index("patched_cell_container") if "patched_cell_container" in metaP.columns else None
dbc_yes["tree_metaPatch"] = mp.loc[dbc_yes.Tube_id, "Tree_first_cl"].values
dbc_yes["ttype_metadata"] = cells_meta.loc[dbc_yes.Tube_id, "Transcriptomic_type"].values
dbc_yes["morph_metadata"] = cells_meta.loc[dbc_yes.Tube_id, "Qualitative_morphology_type"].values
print(dbc_yes[["cell_name_label", "Tree_first_cl", "tree_metaPatch", "ttype_metadata", "morph_metadata"]].to_string())
cnt = dbc_yes.Tree_first_cl.value_counts()
print("DBCs by t-type:", cnt.to_dict())
print("agree with metaPatch:", int((dbc_yes.Tree_first_cl == dbc_yes.tree_metaPatch).sum()), "| with metadata table:",
      int((dbc_yes.Tree_first_cl.str.strip() == dbc_yes.ttype_metadata.str.strip()).sum()), "of", len(dbc_yes))
print("metadata cells with Qualitative_morphology_type DBC:", int((m1.Qualitative_morphology_type == "DBC").sum()))

**Result: reproduced.** The 11 DBCs are 7 SST CALB1 and 4 SST ADGRG6, and the t-types of the DBC table agree with the tree mapping stored with the transcriptomes and with the metadata table for all 11. The tree mapping was not re-run.

## figs7a-dbc-regions

> Six of seven DBCs in SST CALB1 come from temporal cortex and three of four DBCs in SST ADGRG6 from frontal cortex.

Region from the `lobe` column (the NWB files carry no region; see the data check). Over all DBCs and, as a variant, those with an NWB file in the dandiset.

In [ ]:
dbc_yes["lobe"] = cells_meta.loc[dbc_yes.Tube_id, "lobe"].values
dbc_yes["nwb_session"] = cells_meta.loc[dbc_yes.Tube_id, "nwb_session"].values
dbc_yes["in_dandiset"] = dbc_yes.nwb_session.isin(R.session)
print(pd.crosstab(dbc_yes.Tree_first_cl, dbc_yes.lobe, margins=True))
print("with an NWB file in the dandiset:"); print(pd.crosstab(dbc_yes[dbc_yes.in_dandiset].Tree_first_cl, dbc_yes[dbc_yes.in_dandiset].lobe, margins=True))
ca = dbc_yes[dbc_yes.Tree_first_cl.str.contains("CALB1")]; ad = dbc_yes[dbc_yes.Tree_first_cl.str.contains("ADGRG6")]
meas_regions = [int((ca.lobe == "TemL").sum()), len(ca), int((ad.lobe == "FroL").sum()), len(ad)]
print("RESULT figs7a-regions:", meas_regions, "expected [6, 7, 3, 4]")

**Result: reproduced.** Six of 7 SST CALB1 DBCs are from temporal cortex (one occipital) and 3 of 4 SST ADGRG6 DBCs from frontal cortex (one temporal). Two DBCs have no NWB file here. Over the 9 that have one, the counts are 5 of 6 and 2 of 3.

## figs7a-dbc-markers

> Most DBCs strongly express CALB1, only one expresses CALB2, all but one express some SST, and TAC1, NOS1 and NPY expression is mixed.

"Strong" is log2(CPM + 1) > 5 (the criterion's choice); detected is > 0.

In [ ]:
G = X.loc[["CALB1", "CALB2", "SST", "TAC1", "NOS1", "NPY"], dbc_yes.Tube_id].T
G.index = dbc_yes.cell_name_label + " | " + dbc_yes.Tree_first_cl.str.replace("Inh ", "")
print(G.round(2).to_string())
n = len(G)
strong_calb1 = int((G.CALB1 > 5).sum()); calb2 = int((G.CALB2 > 0).sum()); sst = int((G.SST > 0).sum())
mixed = {gname: int((G[gname] > 0).sum()) for gname in ["TAC1", "NOS1", "NPY"]}
print(f"DBCs {n}: CALB1 > 5 in {strong_calb1}; CALB2 > 0 in {calb2}; SST > 0 in {sst}; detected TAC1/NOS1/NPY {mixed}")
print("CALB1 > 3 (variant):", int((G.CALB1 > 3).sum()), "| CALB1 > 7:", int((G.CALB1 > 7).sum()))
ok = strong_calb1 > n / 2 and calb2 == 1 and sst == n - 1 and all(0 < v < n for v in mixed.values())
print("RESULT figs7a-markers:", ok)

**Result: reproduced.** Of 11 DBCs, 7 have CALB1 log2(CPM + 1) > 5, one expresses CALB2, 10 express SST, and TAC1, NOS1 and NPY are detected in 7, 9 and 10. With a cut of 3 for strong CALB1 it is 10 of 11.

## fig4d-calb1-genes-by-morphology

> A UMAP on 253 genes differentially expressed between DBC, MC and sparse SST neurons splits SST CALB1 neurons into three clusters that largely separate the morphology types.

Main reading, the Methods (case iv): pairwise t tests (Seurat's `t`, a Welch test) between DBC, MC and sparse SST cells of SST CALB1, Bonferroni over all genes, adjusted p < 0.05, union of the three pairs. Cells read as (1) all SST CALB1 cells with a morphology call, (2) the culture microglial ones (the code's chunk). Variant, the code's chunk: one-against-rest t tests on the culture microglial cells with genes of median >= 1, counted at Seurat FindAllMarkers' default return threshold (p < 0.01); the code also drops its immune and `keepGenes` lists, which need data not in the repository. Morphology calls are `morpho_call` with the code's fix (cell 893655882 and "SMC" are sparse SST). The UMAP and clusters are run in Python (umap-learn, Louvain of networkx), not Seurat.

In [ ]:
mpc = metaP.copy()
mpc["morpho_call"] = mpc["morpho_call"].astype(str)
mpc.loc[mpc.spec_id.astype(str) == "893655882", "morpho_call"] = "SMC"
mpc.loc[mpc.morpho_call == "SMC", "morpho_call"] = "Sparse SST"
mpc["topLeaf"] = mpc.topLeaf_label.astype(str).str.split(" \\(").str[0]
calb = mpc[mpc.topLeaf == "Inh L1-3 SST CALB1"]
print("SST CALB1 cells", len(calb), "| morphology calls:", calb.morpho_call.value_counts().to_dict(),
      "| condition2 of called cells:", calb[calb.morpho_call.isin(["DBC", "MC", "Sparse SST"])].condition2.value_counts().to_dict())
types = ["DBC", "MC", "Sparse SST"]
def pairwise_de(cols_by_type):
    gs = set()
    out = {}
    ng = X.shape[0]
    for i in range(3):
        for j in range(i + 1, 3):
            A_, B_ = X[cols_by_type[types[i]]].values, X[cols_by_type[types[j]]].values
            if A_.shape[1] < 2 or B_.shape[1] < 2:
                out[(types[i], types[j])] = None
                continue
            with np.errstate(all="ignore"):
                t_, p_ = stats.ttest_ind(A_, B_, axis=1, equal_var=False)
            p_ = np.where(np.isfinite(p_), p_, 1.0)
            sig = X.index[np.minimum(p_ * ng, 1) < 0.05]
            out[(types[i], types[j])] = len(sig)
            gs |= set(sig)
    return gs, out
cols1 = {t_: calb[calb.morpho_call == t_].patched_cell_container.tolist() for t_ in types}
cols2 = {t_: calb[(calb.morpho_call == t_) & calb.condition2.astype(str).str.match(r"culture.*(immune|microglial)")].patched_cell_container.tolist() for t_ in types}
DE = {}
for name, cols in [("1 all called cells", cols1), ("2 culture microglial", cols2)]:
    gs, per = pairwise_de(cols)
    DE[name] = gs
    print(f"reading {name}: cells {[len(cols[t_]) for t_ in types]}, per pair {per}, union {len(gs)}")
c3 = sum(cols2.values(), [])
Xc = X[c3]
Xc = Xc[Xc.median(axis=1) >= 1]
lab3 = np.array([t_ for t_ in types for _ in cols2[t_]])
var = set()
for t_ in types:
    with np.errstate(all="ignore"):
        _, p_ = stats.ttest_ind(Xc.values[:, lab3 == t_], Xc.values[:, lab3 != t_], axis=1, equal_var=False)
    var |= set(Xc.index[np.nan_to_num(p_, nan=1) < 0.01])
print("variant (code's chunk, one against rest, p < 0.01, genes median >= 1):", len(var), "genes")
n_rows = 0
for t_ in types:
    pct_in = (Xc.values[:, lab3 == t_] > 0).mean(1); pct_out = (Xc.values[:, lab3 != t_] > 0).mean(1)
    with np.errstate(all="ignore"):
        _, p_ = stats.ttest_ind(Xc.values[:, lab3 == t_], Xc.values[:, lab3 != t_], axis=1, equal_var=False)
    n_rows += int(((np.nan_to_num(p_, nan=1) < 0.01) & (np.maximum(pct_in, pct_out) >= 0.1)).sum())
print("  as FindAllMarkers counts rows (a gene once per type), with its default min.pct 0.1:", n_rows)
RES_253 = {k: len(v) for k, v in DE.items()}
RES_253["variant"] = len(var)

In [ ]:
import umap, networkx as nx
from sklearn.decomposition import PCA
from sklearn.neighbors import kneighbors_graph
def clusters_on(genes, npcs, nnb, sd):
    Z = X.loc[sorted(genes), calb.patched_cell_container].T
    Z = (Z - Z.mean()) / Z.std(ddof=0).replace(0, np.nan)
    Z = Z.fillna(0).clip(-10, 10)
    pcs = PCA(n_components=min(npcs, Z.shape[0] - 1, Z.shape[1]), random_state=sd).fit_transform(Z)
    emb = umap.UMAP(n_neighbors=nnb, random_state=sd).fit_transform(pcs)
    Gk = nx.from_scipy_sparse_array(kneighbors_graph(emb, 20 if len(emb) > 20 else len(emb) - 1, include_self=False))
    comms = nx.community.louvain_communities(Gk, resolution=1, seed=sd)
    lab = np.zeros(len(emb), int)
    for ci, cm in enumerate(comms):
        lab[list(cm)] = ci
    return lab
mc_ = calb.morpho_call.values
for name, genes, npcs, nnb in [("main, reading 1 genes, 30 PCs", DE["1 all called cells"], 30, 15),
                               ("variant genes, 6 PCs, 5 neighbours", var, 6, 5)]:
    if len(genes) < 3:
        print(name, ": too few genes"); continue
    for off in SEED_OFFSETS:
        lab = clusters_on(genes, npcs, nnb, seed("fig4d-calb1-genes-by-morphology") + off)
        tab = pd.crosstab(mc_, lab).loc[types]
        major = tab.idxmax(axis=1)
        frac = (tab.max(axis=1) / tab.sum(axis=1)).round(2)
        sep = major.nunique() == 3 and (frac > 0.5).all()
        print(f"{name} seed+{off}: clusters {len(set(lab))}, majority cluster {major.to_dict()}, fraction {frac.to_dict()}, separated {sep}")

**Result: not reproduced.** The Methods' pairwise t tests with Bonferroni correction find 2 genes over the 15 called SST CALB1 cells and none over the 10 culture microglial ones. The code's one-against-rest variant finds 431 genes, or 451 rows counted as FindAllMarkers counts them; none of these is within 228 to 278. The clusters cannot be built on 2 genes. On the variant genes three clusters form, but MC and sparse SST cells have their majority in the same cluster under all five seeds, so the three types are not separated.

## PVALB WFDC2, acute against culture

The electrophysiology claims below use the features computed here from the NWB files, labelled through cell-metadata-link.

In [ ]:
W = E[E.ttype == "Inh L2-4 PVALB WFDC2"].copy()
print("PVALB WFDC2 cells with a QC-passing long square sweep:", W.condition.value_counts().to_dict(),
      "| by condition2", W.condition2.value_counts().to_dict(), "| donors", W.Donor.nunique())
MW_W = L.mannwhitney_table(W, FEATS, "condition", "acute", "culture")
print("features tested", len(MW_W), "| significant at BH-FDR < 0.05 across these features:", int((MW_W.p_fdr < 0.05).sum()))

## figs2b-wfdc2-ephys-43-of-84

> In PVALB WFDC2 neurons, 43 of 84 electrophysiological features differ between the acute and culture paradigms.

Per feature a two-sided Mann-Whitney, BH-FDR across features, at FDR p < 0.05. Only the features public IPFX computes are measured here. The count over all 84 is bounded: the missing features add at most their number to the count, and they change the BH adjustment of the others; the bounds take them all non-significant (p = 1) or all significant (p = 0).

In [ ]:
p = MW_W.p.dropna().values
n_missing = len(FEATS84) - len(p)
def bh_count(pvals):
    return int((L.bh(pvals) < 0.05).sum())
lo = bh_count(np.concatenate([p, np.ones(n_missing)]))
hi = bh_count(np.concatenate([p, np.zeros(n_missing)]))
own = bh_count(p)
print(f"features with a test here {len(p)}; missing from the 84: {n_missing}")
print(f"significant over the features computed (BH over them): {own} of {len(p)} ({own / len(p):.1%}); paper 43 of 84 ({43/84:.1%})")
print(f"bounds for the count over 84: {lo} to {hi}; tolerance 38.7 to 47.3")
J84 = W[["specimen_id_x", "condition"]].merge(fx_paper, left_on="specimen_id_x", right_on="specimen_id")
mw_paper = L.mannwhitney_table(J84, FEATS84, "condition", "acute", "culture")
print("variant, on the paper's own table for the same cells (all 84):", int((mw_paper.p_fdr < 0.05).sum()), "| all 93:",
      int((L.mannwhitney_table(J84, FEATS84 + L.CHIRP, "condition", "acute", "culture").p_fdr < 0.05).sum()))
mix = MW_W[["feature", "p"]].set_index("feature").p.reindex(FEATS84)
mix = mix.fillna(mw_paper.set_index("feature").p)
print("variant, computed features with the paper's values for the missing ones:", bh_count(mix.dropna().values))
for c2 in ["culture", "culture_microglial"]:
    sub = W[W.condition2.isin(["acute", "acute_microglial", c2])]
    t2 = L.mannwhitney_table(sub, FEATS, "condition", "acute", "culture")
    print(f"variant, culture cells restricted to {c2}: {int((t2.p_fdr < 0.05).sum())} of {len(t2)}")
inside = 38.7 <= lo and hi <= 47.3
outside = hi < 38.7 or lo > 47.3
print("RESULT figs2b:", "inside" if inside else ("outside" if outside else "undecided"), (lo, hi))

**Result: blocked.** 19 of the 84 features need code the paper has not released, so the count over 84 is bounded between 39 and 61 and the tolerance (38.7 to 47.3) lies inside the bounds. Over the 65 features computed, 41 differ (63%, against the paper's 51%). On the paper's own table, the same cells give 47 of 84.

## fig1l-width-sag-differ

> In PVALB WFDC2 neurons AP width and sag differ between acute and culture slices (sag is higher in culture), while interspike interval and input resistance do not.

Two-sided Mann-Whitney, BH-FDR across the features computed (the 84 cannot all be computed). Width read as rheo, hero, short square and ramp; ISI as mean_isi_rheo and mean_isi_hero.

In [ ]:
t = MW_W.set_index("feature")
show = ["sag", "width_rheo", "width_hero", "width_short_square", "width_ramp", "mean_isi_rheo", "mean_isi_hero", "input_resistance"]
print(t.loc[show, ["n_a", "n_b", "median_a", "median_b", "p", "p_fdr", "rbc"]].to_string(float_format=lambda v: f"{v:.4g}"))
sag_ok = t.loc["sag", "p_fdr"] < 1e-4 and t.loc["sag", "median_b"] > t.loc["sag", "median_a"]
outcomes = {}
for w in ["width_rheo", "width_hero", "width_short_square", "width_ramp"]:
    for isi in ["mean_isi_rheo", "mean_isi_hero"]:
        outcomes[(w, isi)] = bool(sag_ok and t.loc[w, "p_fdr"] < 1e-4 and t.loc[isi, "p_fdr"] >= 0.05 and t.loc["input_resistance", "p_fdr"] >= 0.05)
print("sag higher in culture at FDR p < 1e-4:", bool(sag_ok))
for k, v in outcomes.items():
    print(k, v, "| width direction culture/acute median ratio %.2f" % (t.loc[k[0], "median_b"] / t.loc[k[0], "median_a"]))
for d in sorted(W.Donor.unique())[:0]:
    pass
print("by condition and donor count: acute donors", W[W.condition == "acute"].Donor.nunique(), "culture donors", W[W.condition == "culture"].Donor.nunique())
print("RESULT fig1l: all readings", all(outcomes.values()), "| any", any(outcomes.values()))

**Result: blocked.** The readings disagree. Sag is higher in culture (median 0.284 against 0.110, FDR p = 1.2e-14). AP width is wider in culture at FDR p < 1e-4 for the rheobase, hero and short square sweeps, but not the ramp (FDR p = 0.0077). Input resistance does not differ (FDR p = 0.81). The hero-sweep ISI does not differ (FDR p = 0.75), but the rheobase-sweep ISI does (FDR p = 0.016, 74 cells).

## figs2c-no-change-with-days

> The features that differ in culture already differ at 2 days in culture and do not change with further days in culture.

Pearson's correlation of sag and of AP width with days in culture over culture PVALB WFDC2 cells, BH over the two; and the 2 DIC cells against the acute median.

In [ ]:
Wc = W[W.condition == "culture"]
acute_med = W[W.condition == "acute"][["sag"] + ["width_rheo", "width_hero", "width_short_square", "width_ramp"]].median()
print("culture PVALB WFDC2 cells by days in culture:", Wc.Days_In_culture.value_counts().sort_index().to_dict())
print("all 2 DIC cells in the dandiset:", E[E.Days_In_culture == 2][["ttype", "subclass"]].values.tolist())
res = {}
for w in ["width_rheo", "width_hero", "width_short_square", "width_ramp"]:
    ps, rs = [], []
    for f in ["sag", w]:
        d = Wc[[f, "Days_In_culture"]].dropna()
        r, p = stats.pearsonr(d.Days_In_culture, d[f]); ps.append(p); rs.append(r)
    pf = L.bh(ps)
    res[w] = dict(r_sag=rs[0], p_sag=pf[0], r_width=rs[1], p_width=pf[1])
print(pd.DataFrame(res).T.round(4))
early = Wc[Wc.Days_In_culture == Wc.Days_In_culture.min()]
print("earliest DIC among culture PVALB WFDC2:", int(Wc.Days_In_culture.min()), "cells", len(early))
for f in ["sag", "width_rheo", "width_hero", "width_short_square", "width_ramp"]:
    print(f"  {f}: earliest-DIC median {early[f].median():.4g}, acute median {acute_med[f]:.4g}, culture median {Wc[f].median():.4g}")
for f in MW_W[MW_W.p_fdr < 0.05].feature:
    pass
sigf = MW_W[MW_W.p_fdr < 0.05].feature.tolist()
pp = []
for f in sigf:
    d = Wc[[f, "Days_In_culture"]].dropna()
    pp.append(stats.pearsonr(d.Days_In_culture, d[f])[1] if len(d) > 2 and d[f].nunique() > 1 else np.nan)
print(f"variant, all {len(sigf)} differing features: significant correlations with DIC at BH < 0.05:", int((L.bh(pp) < 0.05).sum()))

**Result: blocked.** It depends on fig1l-width-sag-differ, and its own width readings disagree. Sag does not correlate with days in culture (r = 0.07). Width does not correlate for the rheobase, hero and ramp sweeps, but does for the short square (r = 0.37, FDR p = 0.0015). The one 2 DIC PVALB WFDC2 cell has higher sag and wider APs than the acute median, as the culture group does.

## fig1i-wfdc2-ephys-umap-overlap

> In electrophysiology UMAP space of all 84 features, acute and culture PVALB WFDC2 neurons occupy a similar space.

All cells, the computed features z-scored with missing values at the median; for each PVALB WFDC2 cell, the fraction of its 15 nearest PVALB WFDC2 neighbours from the other condition, against 1000 label shuffles; in feature space and in a default umap-learn embedding.

In [ ]:
Z_all = L.zscore_impute(E[FEATS])
def cross_fraction(Xs, lab, mask, k):
    qidx, nb = L.neighbour_fraction(Xs, lab, k, among=mask, query=mask)
    own = lab[qidx]
    frac = np.array([np.mean(n != o) for n, o in zip(nb, own)])
    return {g_: frac[own == g_].mean() for g_ in np.unique(own)}
def overlap_test(Xs, lab, mask, k, rng, nshuf=1000):
    obs = cross_fraction(Xs, lab, mask, k)
    labm = lab.copy()
    null = {g_: [] for g_ in obs}
    idx = np.where(mask)[0]
    for _ in range(nshuf):
        labm[idx] = rng.permutation(lab[idx])
        cf = cross_fraction(Xs, labm, mask, k)
        for g_ in obs:
            null[g_].append(cf[g_])
    return {g_: (obs[g_], np.median(null[g_])) for g_ in obs}
labW = E.condition.values
maskW = (E.ttype == "Inh L2-4 PVALB WFDC2").values
OUT_1i = {}
for off in SEED_OFFSETS:
    sd = seed("fig1i-wfdc2-ephys-umap-overlap") + off
    emb = L.umap_embed(Z_all, sd)
    rng = np.random.default_rng(seed("fig1i-wfdc2-ephys-umap-overlap", 1) + off)
    for space, Xs in [("features", Z_all.values), ("umap", emb)]:
        for k in ([15, 10, 30] if off == 0 else [15]):
            r_ = overlap_test(Xs, labW, maskW, k, rng, nshuf=1000 if k == 15 else 200)
            OUT_1i[(off, space, k)] = r_
            print(f"seed+{off} {space:8s} k={k}: " + ", ".join(f"{g_} cross {o:.3f} vs shuffle median {n:.3f} (ratio {o/n:.2f})" for g_, (o, n) in r_.items()))
holds = {key: all(o >= 0.5 * n for o, n in v.values()) for key, v in OUT_1i.items() if key[2] == 15}
print("criterion holds (k=15, both spaces) per seed offset:", {off: holds[(off, "features", 15)] and holds[(off, "umap", 15)] for off in SEED_OFFSETS})

**Result: not reproduced.** In feature space, a culture PVALB WFDC2 cell has a mean of 0.19 acute cells among its 15 nearest PVALB WFDC2 neighbours, against 0.45 for shuffled labels (ratio 0.43, under the criterion's 0.5). Acute cells have a ratio of 0.72. In the UMAP embedding the ratios are 0.69 to 0.78 under all five seeds, which passes. With 30 neighbours the feature-space culture ratio is 0.48.

## fig2b-ephys-umap-subclasses

> The four GABAergic subclasses are separable in electrophysiology feature space, with PVALB neurons occupying the most distinct region.

For each cell, the fraction of its 15 nearest neighbours of its own subclass (`Revised_subclass_label`), in feature space and a default umap-learn embedding.

In [ ]:
labS = E.subclass.values
share = pd.Series(labS).value_counts(normalize=True)
OUT_2b = {}
for off in SEED_OFFSETS:
    emb = L.umap_embed(Z_all, seed("fig2b-ephys-umap-subclasses") + off)
    for space, Xs in ([("features", Z_all.values)] if off == 0 else []) + [("umap", emb)]:
        qidx, nb = L.neighbour_fraction(Xs, labS, 15)
        same = pd.Series([np.mean(n == labS[q]) for q, n in zip(qidx, nb)]).groupby(labS[qidx]).mean()
        tab = pd.DataFrame({"same_fraction": same, "share": share, "excess": same - share})
        ok = bool((tab.excess >= 0.2).all() and tab.same_fraction.idxmax() == "PVALB")
        OUT_2b[(off, space)] = ok
        print(f"seed+{off} {space}: ok={ok}"); print(tab.round(3).to_string())
print("by subclass, cells:", pd.Series(labS).value_counts().to_dict())

**Result: not reproduced, narrowly.** PVALB has the highest same-subclass fraction (0.84 in feature space, 0.87 to 0.88 in UMAP). In feature space the VIP excess over its share is 0.193, under the 0.2 the criterion asks. In the UMAP every excess is at least 0.2 under four of the five seeds, the claim's among them (SST 0.199 under the other).

## fig2f-ap-kinetics

> APs of the PVALB subclass have the fastest kinetics, and LAMP5/PAX6 APs have the highest peak amplitude of the four subclasses.

Kruskal-Wallis with Dunn's post hoc (BH-FDR over pairs); short square features first, rheobase sweep second.

In [ ]:
SUBS = ["LAMP5/PAX6/Other", "VIP", "SST", "PVALB"]
def kd_report(df, f, gcol, groups):
    p, dunn, med = L.kruskal_dunn(df, f, gcol, groups)
    print(f"--- {f}: KW p={p:.3g}"); print(med.T.round(5).to_string()); print("Dunn FDR p:"); print(dunn.map(lambda v: f"{v:.2g}").to_string())
    return p, dunn, med
res2f = {}
for sw in ["short_square", "rheo"]:
    ok = True
    for f, best, how in [(f"width_{sw}", "PVALB", "min"), (f"upstroke_downstroke_ratio_{sw}", "PVALB", "min"), (f"peak_deltav_{sw}", "LAMP5/PAX6/Other", "max")]:
        p, dunn, med = kd_report(E, f, "subclass", SUBS)
        m_ = med["median"]
        top_ = m_.idxmin() if how == "min" else m_.idxmax()
        pair_ok = all(dunn.loc[best, o] < 0.05 for o in SUBS if o != best)
        print(f"  extreme group {top_} (expected {best}); Dunn p < 0.05 against every other: {pair_ok}")
        ok &= (top_ == best) and pair_ok
        res2f[(sw, f)] = (top_ == best, pair_ok)
    print("reading", sw, "holds:", ok)

**Result: partially reproduced.** PVALB APs have the narrowest width and the lowest upstroke/downstroke ratio, with Dunn's p < 0.05 against every subclass, for both the short square and the rheobase sweep. LAMP5/PAX6 have the highest median peak in both readings, but not significantly against VIP for the short square (p = 0.08) and against SST for the rheobase sweep (p = 0.061).

## fig2g-sag-by-subclass

> LAMP5/PAX6 and VIP neurons have minimal sag compared with PVALB and SST neurons.

In [ ]:
p, dunn, med = kd_report(E, "sag", "subclass", SUBS)
m_ = med["median"]
pairs = [(a_, b_) for a_ in ["LAMP5/PAX6/Other", "VIP"] for b_ in ["PVALB", "SST"]]
ok = all(m_[a_] < m_[b_] and dunn.loc[a_, b_] < 0.05 for a_, b_ in pairs)
print({f"{a_} vs {b_}": (round(m_[a_], 3), round(m_[b_], 3), float(dunn.loc[a_, b_])) for a_, b_ in pairs})
print("by condition (acute / culture) medians:"); print(E.groupby(["subclass", "condition"]).sag.median().unstack().round(3))
print("RESULT fig2g:", ok)

**Result: reproduced.** The median sag is 0.10 for LAMP5/PAX6 and 0.15 for VIP, against 0.24 for PVALB and 0.30 for SST, and all four pairs differ (Dunn's p <= 1.9e-6).

## fig2h-several-features-distinct

> Several additional electrophysiological features differ between the subclasses.

Kruskal-Wallis across subclasses for every computed feature, BH-FDR across features; counted outside width, upstroke/downstroke ratio, peak and sag features.

In [ ]:
rows = []
for f in FEATS:
    d = E[[f, "subclass"]].dropna()
    samples = [d[d.subclass == s][f].values for s in SUBS]
    if min(len(s) for s in samples) < 2 or d[f].nunique() < 2:
        continue
    rows.append(dict(feature=f, p=stats.kruskal(*samples).pvalue))
kw = pd.DataFrame(rows); kw["p_fdr"] = L.bh(kw.p)
excl = kw.feature.str.contains("width|upstroke_downstroke_ratio|peak_deltav|sag")
other = kw[~excl & (kw.p_fdr < 0.05)].sort_values("p_fdr")
print("features tested", len(kw), "| significant", int((kw.p_fdr < 0.05).sum()), "| outside width/ratio/peak/sag:", len(other))
n_dunn = 0
for f in other.feature:
    _, dunn, _ = L.kruskal_dunn(E, f, "subclass", SUBS)
    n_dunn += int((dunn.values[np.triu_indices(4, 1)] < 0.05).any())
print("of these, with at least one Dunn pair p < 0.05:", n_dunn)
print(other.head(15).to_string(float_format=lambda v: f"{v:.3g}"))

**Result: reproduced.** 59 of 65 features differ across subclasses at FDR < 0.05. 44 of them are outside width, ratio, peak and sag, and 43 of those have a significant Dunn pair.

## fig3c-frzb-ephys-with-pvalb

> Most SST FRZB neurons colocalize with the PVALB subclass in electrophysiology space.

Groups: SST (`subclass_label` SST, not SST FRZB), PVALB (not SST FRZB), SST FRZB (t-type). Features z-scored over SST and PVALB cells; for each SST FRZB cell, its 15 nearest SST or PVALB neighbours.

In [ ]:
frz = E.ttype == "Inh L2-4 SST FRZB"
F3 = E[E.subclass_label.isin(["SST", "PVALB"])].copy()
F3["grp"] = np.where(F3.ttype == "Inh L2-4 SST FRZB", "SST FRZB", F3.subclass_label)
print("groups:", F3.grp.value_counts().to_dict())
Z3 = L.zscore_impute(F3[FEATS])
lab3g = F3.grp.values
for off in SEED_OFFSETS:
    emb = L.umap_embed(Z3, seed("fig3c-frzb-ephys-with-pvalb") + off)
    for space, Xs in ([("features", Z3.values)] if off == 0 else []) + [("umap", emb)]:
        qidx, nb = L.neighbour_fraction(Xs, lab3g, 15, among=lab3g != "SST FRZB", query=lab3g == "SST FRZB")
        pv_frac = np.array([np.mean(n == "PVALB") for n in nb])
        print(f"seed+{off} {space}: SST FRZB cells {len(pv_frac)}, with a PVALB majority {int((pv_frac > 0.5).sum())} ({(pv_frac > 0.5).mean():.2f}), "
              f"mean PVALB fraction {pv_frac.mean():.2f}; PVALB share of reference {np.mean(lab3g[lab3g != 'SST FRZB'] == 'PVALB'):.2f}")

**Result: reproduced.** 72 of 73 SST FRZB cells have a PVALB majority among their 15 nearest SST or PVALB neighbours in feature space, and 71 to 73 in UMAP across the five seeds.

## fig3e-frzb-ap-like-pvalb

> The AP upstroke/downstroke ratio and AP width of SST FRZB neurons align with fast-spiking PVALB neurons and differ from SST neurons.

Kruskal-Wallis over the three groups, Dunn's post hoc (BH-FDR); rheobase, short square and rheobase + 40 pA (hero) sweeps.

In [ ]:
G3 = ["SST", "SST FRZB", "PVALB"]
res3e = {}
for sw in ["rheo", "short_square", "hero"]:
    for f in [f"upstroke_downstroke_ratio_{sw}", f"width_{sw}"]:
        p, dunn, med = kd_report(F3, f, "grp", G3)
        m_ = med["median"]
        nearer = abs(m_["SST FRZB"] - m_["PVALB"]) < abs(m_["SST FRZB"] - m_["SST"])
        res3e[f] = (bool(dunn.loc["SST FRZB", "SST"] < 1e-4), bool(nearer), float(dunn.loc["SST FRZB", "PVALB"]))
        print(f"  FRZB vs SST p<1e-4: {res3e[f][0]}; FRZB nearer PVALB: {nearer}; FRZB vs PVALB p={res3e[f][2]:.3g}")
print("RESULT fig3e:", all(v[0] and v[1] for v in res3e.values()), res3e)

**Result: reproduced.** For the rheobase, short square and hero sweeps, SST FRZB differs from SST in width and upstroke/downstroke ratio (Dunn's p 2e-12 to 7.2e-09 for width, 1.1e-19 to 6.4e-11 for the ratio), and its medians are nearer PVALB. SST FRZB still differs from PVALB in width (p 7.8e-05 to 0.0011) but not in the ratio (p 0.067 to 0.88).

## figs6-sst-features-acute-culture

> The upstroke/downstroke ratio and AP width of SST subclass neurons do not differ between acute and culture recordings.

A claim of absence: two-sided Mann-Whitney, acute against culture, BH-FDR over the two features of a sweep reading; SST (`subclass_label`) with and without SST FRZB.

In [ ]:
anyd = False
for with_frzb in [True, False]:
    S6 = E[(E.subclass_label == "SST") & (with_frzb | (E.ttype != "Inh L2-4 SST FRZB"))]
    for sw in ["rheo", "short_square", "hero"]:
        tt = L.mannwhitney_table(S6, [f"upstroke_downstroke_ratio_{sw}", f"width_{sw}"], "condition", "acute", "culture")
        anyd |= bool((tt.p_fdr < 0.05).any())
        print(f"SST {'with' if with_frzb else 'without'} FRZB, {sw}:"); print(tt[["feature", "n_a", "n_b", "median_a", "median_b", "p", "p_fdr", "rbc"]].to_string(float_format=lambda v: f"{v:.4g}"))
print("RESULT figs6: any difference", anyd)

**Result: reproduced.** Neither feature differs between acute and culture SST cells under any sweep reading, with or without SST FRZB (smallest FDR p 0.22; rank-biserial -0.23 to 0.07).

## fig4c-calb1-ephys-by-morphology

> Within SST CALB1, sparse SST neurons tend to have lower sag, higher adaptation ratio and slightly delayed AP onset than DBCs and MCs, and MCs have the highest AP peak.

SST CALB1 cells with a qualitative morphology type and a file here. Adaptation ratio is `isi_adapt_ratio` (the Methods' first-to-third spike ISI ratio as computed in the helper; its agreement with the paper's table is in cell-metadata-link); `adapt_hero` is shown as a variant only.

In [ ]:
C4 = E[(E.ttype == "Inh L1-3 SST CALB1") & E.Qualitative_morphology_type.notna()].copy()
C4["mt"] = C4.Qualitative_morphology_type
print("cells by type:", C4.mt.value_counts().to_dict(), "| metadata (all, with or without a file):",
      m1[m1.Transcriptomic_type == "Inh L1-3 SST CALB1"].Qualitative_morphology_type.value_counts().to_dict())
cols = ["sag", "isi_adapt_ratio", "latency_rheo", "latency_hero", "peak_deltav_rheo", "peak_deltav_hero", "adapt_hero"]
print(C4[["specimen_id_x", "mt"] + cols].sort_values("mt").to_string(float_format=lambda v: f"{v:.4g}"))
med4 = C4.groupby("mt")[cols].median()
print(med4.round(4).to_string())
sp_, dbc_, mc_ = "Sparse SST", "DBC", "MC"
st = {
    "sag lower": bool(med4.loc[sp_, "sag"] < min(med4.loc[dbc_, "sag"], med4.loc[mc_, "sag"])),
    "adaptation higher": bool(med4.loc[sp_, "isi_adapt_ratio"] > max(med4.loc[dbc_, "isi_adapt_ratio"], med4.loc[mc_, "isi_adapt_ratio"])),
    "latency higher (rheo)": bool(med4.loc[sp_, "latency_rheo"] > max(med4.loc[dbc_, "latency_rheo"], med4.loc[mc_, "latency_rheo"])),
    "latency higher (hero)": bool(med4.loc[sp_, "latency_hero"] > max(med4.loc[dbc_, "latency_hero"], med4.loc[mc_, "latency_hero"])),
    "MC peak highest (rheo)": bool(med4["peak_deltav_rheo"].idxmax() == mc_),
    "MC peak highest (hero)": bool(med4["peak_deltav_hero"].idxmax() == mc_),
}
print(st)
print("variant, adapt_hero (adaptation index): sparse higher than DBC and MC:",
      bool(med4.loc[sp_, "adapt_hero"] > max(med4.loc[dbc_, "adapt_hero"], med4.loc[mc_, "adapt_hero"])))
Jc = C4[["specimen_id_x", "mt"]].merge(fx_paper[["specimen_id", "isi_adapt_ratio"]], left_on="specimen_id_x", right_on="specimen_id")
print("variant, the paper's own isi_adapt_ratio values for these cells, medians:", Jc.groupby("mt").isi_adapt_ratio.median().round(3).to_dict())
for f in ["sag", "isi_adapt_ratio", "latency_rheo", "peak_deltav_rheo"]:
    d = C4[[f, "mt"]].dropna()
    groups = [d[d.mt == t_][f].values for t_ in d.mt.unique() if (d.mt == t_).sum() > 0]
    print(f"  KW {f}: p={stats.kruskal(*groups).pvalue:.3g}")

**Result: reproduced, under an assumption, on 14 cells.** Sparse SST cells have the lowest median sag (0.300 against 0.473 and 0.509), the highest ISI adaptation ratio (1.75 against 1.14 and 1.37) and the longest latency, and MCs the highest peak, under every reading. The ISI adaptation ratio computed here does not match the paper's definition (27% of cells within 10%). The paper's own values for these cells order the types the same way. No Kruskal-Wallis test of adaptation or latency is significant (p = 0.16 and 0.11).

## fig5c-dbc-higher-sag

> A higher sag ratio is a consistent and prominent feature of all DBCs.

DBCs from `dbc_calb1_ADGRG6.csv`. Comparison (a) the other cells of SST CALB1 and SST ADGRG6, (b) MC and sparse SST cells of SST CALB1.

In [ ]:
E["is_dbc"] = E.patched_cell_container.isin(dbc_yes.Tube_id)
two = E[E.ttype.isin(["Inh L1-3 SST CALB1", "Inh L3-5 SST ADGRG6"])]
dbc_e = two[two.is_dbc]
print("DBCs with a file and sag:", int(dbc_e.sag.notna().sum()), "of", len(dbc_yes), "| sag values", dbc_e.sag.round(3).tolist(), dbc_e.ttype.str[4:].tolist())
comp_a = two[~two.is_dbc]
comp_b = E[(E.ttype == "Inh L1-3 SST CALB1") & E.Qualitative_morphology_type.isin(["MC", "Sparse SST"])]
for name, cmp_ in [("a", comp_a), ("b", comp_b)]:
    cm = cmp_.sag.median(); dm = dbc_e.sag.median()
    u, p = stats.mannwhitneyu(dbc_e.sag.dropna(), cmp_.sag.dropna(), alternative="two-sided")
    print(f"reading {name}: comparison n={cmp_.sag.notna().sum()} median {cm:.3f}; DBC median {dm:.3f}; DBCs above comparison median "
          f"{int((dbc_e.sag > cm).sum())} of {int(dbc_e.sag.notna().sum())}; Mann-Whitney p={p:.3g}")

**Result: partially reproduced.** The DBC median sag (0.473, 8 DBCs) is above both comparison groups. All 8 DBCs are above the median of the MC and sparse SST cells (0.358, reading b). Only 6 of 8 are above the median of the other SST CALB1 and ADGRG6 cells (0.389, reading a; Mann-Whitney p = 0.028).

## fig5d-dbc-ephys-intermingled

> In electrophysiology space, DBCs of the SST CALB1 and SST ADGRG6 types are intermingled with other SST neurons.

SST subclass cells (`subclass_label`), features z-scored over them; for each DBC, the fraction of its 10 nearest SST neighbours that are DBCs.

In [ ]:
S5 = E[E.subclass_label == "SST"].copy()
Z5 = L.zscore_impute(S5[FEATS]); lab5 = np.where(S5.is_dbc, "DBC", "other")
share5 = np.mean(lab5 == "DBC")
for off in SEED_OFFSETS:
    emb = L.umap_embed(Z5, seed("fig5d-dbc-ephys-intermingled") + off)
    for space, Xs in ([("features", Z5.values)] if off == 0 else []) + [("umap", emb)]:
        qidx, nb = L.neighbour_fraction(Xs, lab5, 10, query=lab5 == "DBC")
        fr = np.array([np.mean(n == "DBC") for n in nb])
        print(f"seed+{off} {space}: DBCs {len(fr)}, mean DBC fraction among 10 neighbours {fr.mean():.3f}; DBC share {share5:.3f}; "
              f"ratio {fr.mean() / share5:.2f}; holds {fr.mean() <= 2 * share5}")

**Result: not reproduced.** The 9 DBCs have a mean of 0.133 DBCs among their 10 nearest SST neighbours in feature space, 3.4 times their share (0.039). In UMAP the ratio is 2.6 under the claim's seed and 2.3 to 3.7 over the five seeds, above the criterion's 2 throughout.

## figs5-vip-enhancer-enrichment

> Neurons labelled with the VIP enhancer AAV CN2039 did not map only to VIP types but were strongly enriched for VIP types.

By `Revised_subclass_label`; comparison (a) all cells without CN2039, (b) CN1390-only cells; two-sided Fisher exact test (the criterion's choice). Over all labelled cells, and over those with a file in the dandiset as a variant.

In [ ]:
for name, M in [("all labelled cells", m1), ("cells with a file here", m1[m1.nwb_session.isin(R.session)])]:
    vip = M.Revised_subclass_label == "VIP"
    cn = M.Virus == "CN2039"
    print(f"{name}: CN2039 cells {int(cn.sum())}, VIP among them {int((vip & cn).sum())} ({(vip & cn).sum() / cn.sum():.2f}); subclasses",
          M[cn].Revised_subclass_label.value_counts().to_dict())
    for rd, other in [("a", ~cn), ("b", M.Virus == "CN1390")]:
        tab = [[int((vip & cn).sum()), int((~vip & cn).sum())], [int((vip & other).sum()), int((~vip & other).sum())]]
        o, p = stats.fisher_exact(tab)
        print(f"  reading {rd}: comparison VIP fraction {tab[1][0] / sum(tab[1]):.3f} (n={sum(tab[1])}), odds ratio {o:.2f}, Fisher p={p:.3g}")

**Result: reproduced.** 19 of 30 CN2039 cells are VIP (0.63), against 0.138 of other cells (Fisher p = 1.7e-09) and 0.053 of CN1390 cells (p = 1e-14). The other 11 are PVALB, SST or LAMP5/PAX6.

## Morphology features (elsewhere)

The morphology claims are `data: elsewhere`. They are tested on the paper's morphology feature table (`LeeDalley_morpho_features.csv`, 140 reconstructions, at commit 5eb357a); the features are not recomputed from the reconstructions at the Brain Image Library.

In [ ]:
MF = morph.drop(columns=[c for c in morph.columns if c.startswith("Unnamed")]).merge(
    m1, left_on="specimen_id", right_on="specimen_id_x", how="left")
MFEATS = [c for c in morph.columns if c not in ("specimen_id",) and not c.startswith("Unnamed")]
print("reconstructions", len(MF), "| labelled", int(MF.specimen_id_x.notna().sum()), "| features", len(MFEATS))
print("by subclass:", MF.Revised_subclass_label.value_counts().to_dict())
print("soma depth range (um):", MF.soma_aligned_dist_from_pia.min().round(1), MF.soma_aligned_dist_from_pia.max().round(1))

## fig1g-wfdc2-morphology-unchanged

> Fifty morphological features of depth-matched PVALB WFDC2 neurons do not differ between culture (n = 9) and acute (n = 10) slices.

Mann-Whitney per feature, BH over the 50 features (the 51 columns less soma depth, used for matching). Readings: (a) all PVALB WFDC2 reconstructions, (b) depth-matched, here the cells inside the depth range both groups share, (c) nearest-depth pairs.

In [ ]:
MW_ = MF[MF.Transcriptomic_type == "Inh L2-4 PVALB WFDC2"].copy()
F50 = [c for c in MFEATS if c != "soma_aligned_dist_from_pia"]
print("PVALB WFDC2 reconstructions:", MW_.condition.value_counts().to_dict(), "| features", len(F50))
print(MW_.groupby("condition").soma_aligned_dist_from_pia.describe().round(1))
lo_d = MW_.groupby("condition").soma_aligned_dist_from_pia.min().max(); hi_d = MW_.groupby("condition").soma_aligned_dist_from_pia.max().min()
rb = MW_[MW_.soma_aligned_dist_from_pia.between(lo_d, hi_d)]
cu = MW_[MW_.condition == "culture"]; ac = MW_[MW_.condition == "acute"].copy()
pick = []
for _, r_ in cu.sort_values("soma_aligned_dist_from_pia").iterrows():
    j = (ac.soma_aligned_dist_from_pia - r_.soma_aligned_dist_from_pia).abs().drop(pick, errors="ignore").idxmin(); pick.append(j)
rc = pd.concat([cu, ac.loc[pick]])
for name, d in [("a all", MW_), ("b shared depth range", rb), ("c nearest-depth pairs", rc)]:
    tt = L.mannwhitney_table(d, F50, "condition", "acute", "culture")
    print(f"reading {name}: acute {int((d.condition == 'acute').sum())}, culture {int((d.condition == 'culture').sum())}; "
          f"features with FDR p < 0.05: {int((tt.p_fdr < 0.05).sum())}; smallest FDR p {tt.p_fdr.min():.3g}; smallest raw p {tt.p.min():.3g} ({tt.loc[tt.p.idxmin(), 'feature']})")

**Result: reproduced.** No morphology feature differs at FDR p < 0.05 between acute and culture PVALB WFDC2 cells under any reading (smallest FDR p 0.09 to 0.20). The table holds 16 acute and 10 culture reconstructions, against the paper's 10 and 9.

## fig2c-morph-umap-subclasses

> The four subclasses are separable in morphological feature space, PVALB the most distinct.

As fig2b-ephys-umap-subclasses on the morphology features of the 140 reconstructed neurons.

In [ ]:
ZM = L.zscore_impute(MF[MFEATS]); labM = MF.Revised_subclass_label.values
shareM = pd.Series(labM).value_counts(normalize=True)
for off in SEED_OFFSETS:
    emb = L.umap_embed(ZM, seed("fig2c-morph-umap-subclasses") + off)
    for space, Xs in ([("features", ZM.values)] if off == 0 else []) + [("umap", emb)]:
        qidx, nb = L.neighbour_fraction(Xs, labM, 15)
        same = pd.Series([np.mean(n == labM[q]) for q, n in zip(qidx, nb)]).groupby(labM[qidx]).mean()
        tab = pd.DataFrame({"same_fraction": same, "share": shareM, "excess": same - shareM})
        ok = bool((tab.excess >= 0.2).all() and tab.same_fraction.idxmax() == "PVALB")
        print(f"seed+{off} {space}: ok={ok}"); print(tab.round(3).to_string())

**Result: not reproduced.** PVALB has the highest same-subclass fraction in both spaces (0.78 and 0.73). LAMP5/PAX6 exceed their share by only 0.10 in feature space, and SST by 0.175 in the UMAP of the claim's seed (0.14 to 0.21 over the five seeds).

## fig2e-lamp5-shortest-extent

> LAMP5/PAX6 neurons have the shortest soma-to-branch-tip Euclidean distance for both axon and dendrite.

In [ ]:
MF["subclass"] = MF.Revised_subclass_label
for f in ["axon_max_euclidean_distance", "basal_dendrite_max_euclidean_distance"]:
    p, dunn, med = kd_report(MF, f, "subclass", SUBS)
    print(f"  lowest median: {med['median'].idxmin()}")

**Result: reproduced.** LAMP5/PAX6 have the lowest median axon (434 um) and dendrite (171 um) maximum Euclidean distance, with Kruskal-Wallis p = 4.6e-06 and 2.4e-04.

## fig3g-frzb-morphology-like-pvalb

> SST FRZB neurons overlap more with PVALB than with SST neurons in morphology space, their horizontal axon extent and axon versus dendrite earth mover's distance resemble PVALB basket cells, and in co-clustering they fall more often into PVALB-dominated types.

Kruskal-Wallis with Dunn (BH-FDR) for `axon_extent_x` and `axon_emd_with_basal_dendrite`; co-clustering as the Methods (95% subsamples, shared nearest neighbour graph, fast-greedy, 500 resamplings, Ward linkage on the co-clustering matrix, cut at six clusters), with 10 and 20 graph neighbours. A cluster is PVALB- or SST-dominated by the majority `subclass_label` of its cells other than SST FRZB.

In [ ]:
M3 = MF[MF.subclass_label.isin(["SST", "PVALB"])].copy()
M3["grp"] = np.where(M3.Transcriptomic_type == "Inh L2-4 SST FRZB", "SST FRZB", M3.subclass_label)
print("groups:", M3.grp.value_counts().to_dict())
r3g = {}
for f in ["axon_extent_x", "axon_emd_with_basal_dendrite"]:
    p, dunn, med = kd_report(M3, f, "grp", G3)
    m_ = med["median"]
    r3g[f] = (bool(abs(m_["SST FRZB"] - m_["PVALB"]) < abs(m_["SST FRZB"] - m_["SST"])), float(dunn.loc["SST", "SST FRZB"]))
print("nearer PVALB, Dunn SST vs FRZB p:", r3g)
import igraph as ig
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
Z3m = L.zscore_impute(M3[MFEATS]).values
def snn_fastgreedy(Xs, k):
    from sklearn.neighbors import NearestNeighbors
    _, ind = NearestNeighbors(n_neighbors=k + 1).fit(Xs).kneighbors(Xs)
    sets = [set(r) for r in ind]
    edges, w = [], []
    for i in range(len(Xs)):
        for j in ind[i][1:]:
            if i < j or i not in ind[j]:
                s = len(sets[i] & sets[j]) / len(sets[i] | sets[j])
                if s > 0 and (min(i, j), max(i, j)) not in edges:
                    edges.append((min(i, j), max(i, j))); w.append(s)
    g_ = ig.Graph(n=len(Xs), edges=edges); g_.es["weight"] = w
    return np.array(g_.community_fastgreedy(weights="weight").as_clustering().membership)
for off in SEED_OFFSETS:
    rng = np.random.default_rng(seed("fig3g-frzb-morphology-like-pvalb") + off)
    for k in [10, 20]:
        n = len(Z3m); co = np.zeros((n, n)); both = np.zeros((n, n))
        for _ in range(500):
            idx = np.sort(rng.choice(n, int(round(0.95 * n)), replace=False))
            lab_ = snn_fastgreedy(Z3m[idx], k)
            same = (lab_[:, None] == lab_[None, :]).astype(float)
            co[np.ix_(idx, idx)] += same; both[np.ix_(idx, idx)] += 1
        C = co / np.maximum(both, 1)
        dist = 1 - C; np.fill_diagonal(dist, 0)
        cl = fcluster(linkage(squareform(dist, checks=False), "ward"), 6, "maxclust")
        M3["cl"] = cl
        dom = M3[M3.grp != "SST FRZB"].groupby("cl").subclass_label.agg(lambda s: s.value_counts().idxmax() if len(s) else "none")
        fz = M3[M3.grp == "SST FRZB"].cl.map(dom).fillna("none").value_counts().to_dict()
        if off == 0:
            print(f"k={k}:"); print(pd.crosstab(M3.cl, M3.grp))
        print(f"seed+{off} k={k}: SST FRZB cells by cluster dominance {fz}")

**Result: reproduced.** SST FRZB medians of axon_extent_x (855) and axon_emd_with_basal_dendrite (15.7) are near PVALB (910, 15.8) and differ from SST (Dunn's p 0.0008 and 0.0006). In co-clustering, 12 of 14 SST FRZB cells fall in PVALB-dominated clusters, for 10 and 20 neighbours and all five seeds.

## fig1h-wfdc2-morph-umap-overlap

> In morphological UMAP space, culture and acute PVALB WFDC2 neurons occupy similar space in the PVALB-rich portion.

The measure of fig1i-wfdc2-ephys-umap-overlap with 5 neighbours on all reconstructions (z-scored morphology features); PVALB-rich part: most of each PVALB WFDC2 cell's 5 nearest neighbours among all cells are PVALB, for most cells of each group. Groups: all PVALB WFDC2 reconstructions, and the shared-depth subset of fig1g.

In [ ]:
labC = MF.condition.values
labSub = MF.Revised_subclass_label.values
for setname, ids in [("all PVALB WFDC2", MW_.specimen_id), ("shared depth range", rb.specimen_id)]:
    maskM = MF.specimen_id.isin(ids).values
    for off in SEED_OFFSETS:
        emb = L.umap_embed(ZM, seed("fig1h-wfdc2-morph-umap-overlap") + off)
        rng = np.random.default_rng(seed("fig1h-wfdc2-morph-umap-overlap", 1) + off)
        for space, Xs in ([("features", ZM.values)] if off == 0 else []) + [("umap", emb)]:
            ov = overlap_test(Xs, labC, maskM, 5, rng)
            qidx, nb = L.neighbour_fraction(Xs, labSub, 5, query=maskM)
            pv = np.array([np.mean(n == "PVALB") for n in nb]) > 0.5
            rich = {g_: pv[labC[qidx] == g_].mean() for g_ in ["acute", "culture"]}
            ok = all(o >= 0.5 * n_ for o, n_ in ov.values()) and all(v > 0.5 for v in rich.values())
            print(f"{setname} seed+{off} {space}: " + ", ".join(f"{g_} cross {o:.2f} vs shuffle {n_:.2f}" for g_, (o, n_) in ov.items())
                  + f"; PVALB-majority fraction {({k: round(v, 2) for k, v in rich.items()})}; holds {ok}")

**Result: reproduced.** Acute and culture PVALB WFDC2 reconstructions are mixed (cross-group fractions 0.30 to 0.72 against shuffled 0.40 to 0.66). In both readings, both spaces and all five seeds, at least 88% of these cells have a PVALB majority among their 5 nearest neighbours.

## fig5d-dbc-morph-umap

> In morphological UMAP space, DBCs of the SST CALB1 and SST ADGRG6 types cluster near each other.

SST subclass reconstructions (`subclass_label`), z-scored; for each DBC, the fraction of its 5 nearest SST neighbours that are DBCs; reproduced when the mean is at least twice the DBC share.

In [ ]:
MS = MF[MF.subclass_label == "SST"].copy()
MS["is_dbc"] = MS.patched_cell_container.isin(dbc_yes.Tube_id)
ZS = L.zscore_impute(MS[MFEATS]); labD = np.where(MS.is_dbc, "DBC", "other"); shareD = np.mean(labD == "DBC")
ttD = MS.Transcriptomic_type.values
print("SST reconstructions", len(MS), "| DBCs among them", int(MS.is_dbc.sum()))
for off in SEED_OFFSETS:
    emb = L.umap_embed(ZS, seed("fig5d-dbc-morph-umap") + off)
    for space, Xs in ([("features", ZS.values)] if off == 0 else []) + [("umap", emb)]:
        qidx, nb = L.neighbour_fraction(Xs, labD, 5, query=labD == "DBC")
        fr = np.array([np.mean(n == "DBC") for n in nb])
        _, nbt = L.neighbour_fraction(Xs, ttD, 5, among=labD == "DBC", query=labD == "DBC")
        own_tt = np.mean([np.mean(n == ttD[q]) for q, n in zip(qidx, nbt)])
        print(f"seed+{off} {space}: mean DBC fraction {fr.mean():.3f}, share {shareD:.3f}, ratio {fr.mean() / shareD:.2f}, holds {fr.mean() >= 2 * shareD}; "
              f"own t-type among DBC neighbours {own_tt:.2f}")

**Result: reproduced, but seed-sensitive.** In feature space DBCs have a mean of 0.78 DBCs among their 5 nearest SST neighbours, 3.5 times their share (0.22). In the UMAP of the claim's seed the ratio is 2.02, and over the five seeds 1.54 to 2.59, under 2 for two of them.

## Summary

| Claim | Outcome |
|---|---|
| cell-metadata-link | reproduced |
| dataset-composition | partially reproduced |
| fig1b-more-genes-in-culture | reproduced |
| figs1a-genes-across-days | reproduced |
| fig1d-nms-no-difference | reproduced |
| fig1c-marker-expression-correlated | reproduced (assumption) |
| figs2d-seven-channel-genes | blocked (other) |
| fig5a-dbc-two-ttypes | reproduced |
| figs7a-dbc-regions | reproduced |
| figs7a-dbc-markers | reproduced |
| fig4d-calb1-genes-by-morphology | not reproduced |
| figs2b-wfdc2-ephys-43-of-84 | blocked (methods underspecified) |
| fig1l-width-sag-differ | blocked (methods underspecified) |
| figs2c-no-change-with-days | blocked (methods underspecified) |
| fig1i-wfdc2-ephys-umap-overlap | not reproduced |
| fig2b-ephys-umap-subclasses | not reproduced |
| fig2f-ap-kinetics | partially reproduced |
| fig2g-sag-by-subclass | reproduced |
| fig2h-several-features-distinct | reproduced |
| fig3c-frzb-ephys-with-pvalb | reproduced |
| fig3e-frzb-ap-like-pvalb | reproduced |
| figs6-sst-features-acute-culture | reproduced |
| fig4c-calb1-ephys-by-morphology | reproduced (assumption) |
| fig5c-dbc-higher-sag | partially reproduced |
| fig5d-dbc-ephys-intermingled | not reproduced |
| figs5-vip-enhancer-enrichment | reproduced |
| fig1g-wfdc2-morphology-unchanged | reproduced |
| fig2c-morph-umap-subclasses | not reproduced |
| fig2e-lamp5-shortest-extent | reproduced |
| fig3g-frzb-morphology-like-pvalb | reproduced |
| fig1h-wfdc2-morph-umap-overlap | reproduced |
| fig5d-dbc-morph-umap | reproduced (seed-sensitive) |
| Seurat, snRNA-seq, MERFISH, bouton, reconstruction and mouse claims (13) | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. 19 of the 84 electrophysiology features need code the paper has not released, so the multivariate measures use the 65 that public IPFX computes. Cell-level QC (seal, access resistance) is not recomputed. The sweep QC of the Methods is applied. The UMAP claims are judged by the neighbour measures the claims file fixes, not by the paper's plots. The morphology claims use the paper's feature table. Seurat analyses are not run.

What was found about the dataset. The files can be linked to the paper's tables through `nwb_id`, and the features computed from them match the paper's table for most cells. 16 files have no row in the paper's metadata, 20 rows with electrophysiology have no file, and the archive's summary gives 616 cells for 706 one-cell files. 79 files have sweeps sampled at 200 kHz where the Methods state 50 kHz. The files carry no cell type, region, layer, condition or days in culture, so every label comes from the code repository. Species is written two ways.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")